<a href="https://colab.research.google.com/github/1900690/OCR/blob/main/OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 事前準備
#@markdown　このセルをはじめに実行してください
# パッケージのインストール
!pip install yomitoku nest_asyncio pandas lxml -q

import nest_asyncio
nest_asyncio.apply()

import os
import cv2
import io
import numpy as np
import pandas as pd
from google.colab import files
from google.colab.patches import cv2_imshow
from yomitoku import DocumentAnalyzer
from yomitoku.data.functions import load_pdf, load_image

# モデルの初期化
analyzer = DocumentAnalyzer(visualize=True, device="cuda")

In [ ]:
# @title OCR表抽出アプリ (改訂版)

# --- フォーム設定項目 ---
# @markdown ### 1. 解析・出力設定
# @markdown 上部の切り取り％を設定してください
TOP_CROP_RATIO = 0.06 # @param {type:"number"}
DISPLAY_WIDTH = 800
# @markdown ### 2. ファイル取得方法
# @markdown ファイルをアップロードするかサンプルファイルで分析するかを選択してください
SOURCE_TYPE = "Upload File" # @param ["Upload File", "Use Sample"]
SAMPLE_URL = "https://github.com/1900690/OCR/raw/main/SKM_C301i26051411120.pdf"
# @markdown ### 3. ダウンロード設定
# @markdown チェックを入れると、自動でダウンロードを開始します。
DOWNLOAD_CSV = True # @param {type:"boolean"}
DOWNLOAD_IMAGE = True # @param {type:"boolean"}

# ----------------
import os
import io
import logging
import numpy as np
import pandas as pd
import cv2
from google.colab import files
from google.colab.patches import cv2_imshow

# --- 徹底的なログ消去設定 ---
# 1. "yomitoku" という名前を含むすべてのロガーを対象にする
for logger_name in logging.root.manager.loggerDict:
    if "yomitoku" in logger_name:
        target_logger = logging.getLogger(logger_name)

        # エラー以外は表示しない
        target_logger.setLevel(logging.ERROR)

        # 既存の出力設定（ハンドラ）をすべて削除する
        for handler in target_logger.handlers[:]:
            target_logger.removeHandler(handler)

        # 親ロガー（システム全体）への報告を遮断
        target_logger.propagate = False

# 2. まだ生成されていないサブロガー対策として、基本名も個別に封じる
base_logger = logging.getLogger("yomitoku")
base_logger.setLevel(logging.ERROR)
base_logger.propagate = False


##################

file_path = ""

# 1. ファイルの準備
if SOURCE_TYPE == "Use Sample":
    file_path = "sample_data.pdf"
    print("サンプルファイルをダウンロード中...")
    !wget -q {SAMPLE_URL} -O {file_path}
    print(f"✅ サンプルファイルの準備完了: {file_path}")
else:
    print("解析したいファイル（PDFまたは画像）を選択してください。")
    uploaded = files.upload()
    if uploaded:
        file_path = list(uploaded.keys())[0]
        print(f"✅ アップロード完了: {file_path}")
    else:
        print("ファイルが選択されませんでした。処理を中断します。")

if file_path:
    # 2. ファイル読み込み
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.pdf':
        imgs = load_pdf(file_path) # 既存のload_pdf関数を使用
    elif ext in ['.png', '.jpg', '.jpeg']:
        imgs = load_image(file_path) # 既存のload_image関数を使用
    else:
        raise ValueError(f"対応していない拡張子です: {ext}")

    # --- 条件③：複数ページに対応するためのループ ---
    for i, img in enumerate(imgs):
        page_num = i + 1
        print(f"\n--- ページ {page_num} の処理を開始 ---")

        if not isinstance(img, np.ndarray):
            img = np.array(img)

        # 3. トリミング処理
        h, w = img.shape[:2]
        start_row = int(h * TOP_CROP_RATIO)
        processed_img = img[start_row:h, 0:w]

        # 4. 解析実行
        print(f"ページ {page_num} を解析中...")
        results, ocr_vis, layout_vis = analyzer(processed_img) # 既存のanalyzerを使用

        # 5. HTMLおよびCSVへの変換
        out_html_path = f"result_page_{page_num}.html"
        html_content = results.to_html(out_html_path, img=processed_img)
        with open(out_html_path, "w", encoding="utf-8") as f:
            f.write(html_content)

        try:
            tables = pd.read_html(io.StringIO(html_content), header=0)
            if len(tables) > 0:
                for t_idx, df in enumerate(tables):
                    csv_path = f"result_p{page_num}_table_{t_idx}.csv"
                    df.to_csv(csv_path, index=False, encoding="utf-8-sig", float_format='%.0f')
                    print(f"✅ CSV作成完了: {csv_path}")

                    # 条件②：チェックがある場合のみCSVダウンロード
                    if DOWNLOAD_CSV:
                        files.download(csv_path)
            else:
                print(f"ページ {page_num} に表は見つかりませんでした。")
        except Exception as e:
            print(f"CSV変換エラー (Page {page_num}): {e}")

        # 6. 可視化結果の表示と保存
        # --- 条件①：表示サイズを一定にする（リサイズ） ---
        vis_h, vis_w = ocr_vis.shape[:2]
        aspect_ratio = vis_h / vis_w
        target_height = int(DISPLAY_WIDTH * aspect_ratio)
        resized_vis = cv2.resize(ocr_vis, (DISPLAY_WIDTH, target_height))

        img_filename = f"result_ocr_page_{page_num}.jpg"
        cv2.imwrite(img_filename, ocr_vis) # 保存は元の解像度で

        print(f"\n【ページ {page_num} OCR検出結果】")
        cv2_imshow(resized_vis)

        # 条件②：チェックがある場合のみ画像ダウンロード
        if DOWNLOAD_IMAGE:
            print(f"画像をダウンロードします: {img_filename}")
            files.download(img_filename)

    print("\n✨ 全ページの処理が完了しました。")